In [6]:
import hail as hl
import os
import pandas as pd
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import samples
from ukb_utils import variants
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa014.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [11]:
# Load phased data
mt = hl.import_vcf('/well/lindgren/UKBIOBANK/nbaya/wes_200k/phase_ukb_wes/data/phased/non_singleton/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz', force_bgz = True)

In [25]:
# load trio data
trios = hl.read_matrix_table('data/qc/ukb_wes_200k_switch_errors_chr20.mt/')

In [31]:
trios.proband_entry.GT.show()

,
locus,alleles
locus<GRCh38>,array<str>
chr20:87662,"[""A"",""G""]"
chr20:87675,"[""C"",""T""]"
chr20:87693,"[""C"",""T""]"
chr20:87695,"[""T"",""G""]"
chr20:87699,"[""T"",""A""]"
chr20:87706,"[""A"",""G""]"
chr20:87716,"[""A"",""G""]"
chr20:87722,"[""A"",""T""]"


In [54]:
app_id = 11867
RESOURCES_DIR = "/well/lindgren/UKBIOBANK/nbaya/resources"

def get_fam_path(app_id = 11867, wes_200k_only=False, relateds_only=True):
    return f'{RESOURCES_DIR}/ukb{app_id}{"_wes_200k" if wes_200k_only else ""}_pedigree{"_related" if relateds_only else ""}.fam'


In [12]:
fam_path = samples.get_fam_path(app_id=11867, wes_200k_only=False, relateds_only = False)

ht = hl.import_table(fam_path).key_by('IID')




2021-11-15 11:05:00 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (not specified)
  Loading field 'IID' as type str (not specified)
  Loading field 'PAT' as type str (not specified)
  Loading field 'MAT' as type str (not specified)
  Loading field 'SEX' as type str (not specified)
  Loading field 'PHEN' as type str (not specified)


In [19]:
mt.filter_cols(~hl.is_defined(ht[mt.col_key])).count()

2021-11-15 11:07:53 Hail: INFO: Coerced prefix-sorted dataset


(60590, 3467)

In [4]:
ped = hl.Pedigree.read(fam_path)

2021-11-15 10:51:35 Hail: WARN: Found 1 samples with missing sex information (not 1 or 2).
 Missing samples: [{'IID'}]


In [9]:
ped.complete_trios()

[Trio(s='IID', fam_id='FID', pat_id='PAT', mat_id='MAT', is_female=None),
 Trio(s='1079534', fam_id='327', pat_id='3198027', mat_id='1536567', is_female=True),
 Trio(s='2137986', fam_id='327', pat_id='3198027', mat_id='1536567', is_female=True),
 Trio(s='1087376', fam_id='364', pat_id='1012671', mat_id='1656605', is_female=True),
 Trio(s='5150887', fam_id='364', pat_id='1012671', mat_id='1656605', is_female=True),
 Trio(s='1214832', fam_id='888', pat_id='5532468', mat_id='5711509', is_female=False),
 Trio(s='1520470', fam_id='888', pat_id='5532468', mat_id='5711509', is_female=False),
 Trio(s='1800233', fam_id='3311', pat_id='5938029', mat_id='3545102', is_female=True),
 Trio(s='3134390', fam_id='3311', pat_id='5938029', mat_id='3545102', is_female=False),
 Trio(s='2056563', fam_id='4314', pat_id='2357088', mat_id='4051261', is_female=True),
 Trio(s='5127307', fam_id='4314', pat_id='2357088', mat_id='4051261', is_female=False),
 Trio(s='2367425', fam_id='5515', pat_id='5652582', mat_id

In [20]:
#mt = mt.filter_cols(~hl.is_defined(ht[mt.col_key]))
ht = hl.trio_matrix(mt, ped, complete_trios = True)
ht.count()


2021-11-15 11:12:02 Hail: INFO: Coerced prefix-sorted dataset
2021-11-15 11:13:59 Hail: INFO: Coerced prefix-sorted dataset


(60590, 92)

In [24]:
phased_trio_dataset = hl.experimental.phase_trio_matrix_by_transmission(ht)

In [28]:
#phased_trio_dataset.describe()

In [29]:
phased_trio_dataset.count()

2021-11-15 11:53:22 Hail: INFO: Coerced prefix-sorted dataset


(60590, 92)

In [33]:
# load dataset
ht = samples.get_fam(app_id=11867, wes_200k_only=False)
trios = ht.filter((ht.PAT != '0') & (ht.MAT != '0'))

2021-11-14 10:47:24 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)


In [113]:



# load matrixtable and subset to trios
dataset = qc.get_table('data/unphased/post-qc/ukb_wes_200k_filtered_chr21.vcf.bgz','vcf')
dataset = dataset.filter_cols(hl.is_defined(trios[dataset.s].FID))

In [93]:
# get pedigree
fam_path = get_fam_path(app_id=11867,wes_200k_only=False,relateds_only=False)
ped = hl.Pedigree.read(fam_path)

2021-11-10 14:45:31 Hail: WARN: Found 1 samples with missing sex information (not 1 or 2).
 Missing samples: [{'IID'}]


In [114]:
trio_dataset = hl.trio_matrix(dataset, pedigree, complete_trios=True)

2021-11-10 15:11:08 Hail: INFO: Coerced sorted dataset


In [115]:
phased_trio_dataset = hl.experimental.phase_trio_matrix_by_transmission(trio_dataset)

In [127]:
#phased_trio_dataset.describe()
dataset = qc.get_table('data/unphased/post-qc/ukb_wes_200k_filtered_chr21.vcf.bgz','vcf')

2021-11-10 15:21:53 Hail: INFO: Coerced sorted dataset


In [117]:
#phased_trio_dataset = phased_trio_dataset.annotate_entries(
#    GT_flip = phased_trio_dataset.proband_entry.PBT_GT != phased_trio_dataset.proband_entry.GT
#)

In [118]:
result_mt = phased_trio_dataset.annotate_rows(switch_errors=hl.agg.sum(phased_trio_dataset.GT_flip))

In [121]:
result_mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    'id': str
    'proband': struct {
        s: str
    }
    'father': struct {
        s: str
    }
    'mother': struct {
        s: str
    }
    'is_female': bool
    'fam_id': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AQ: array<int32>, 
        AC: int32, 
        AN: int32, 
        ExcessHet: float64
    }
    'switch_errors': int64
----------------------------------------
Entry fields:
    'proband_entry': struct {
        GT: call, 
        DP: int32, 
        AD: array<int32>, 
        GQ: int32, 
        PL: array<int32>, 
        PBT_GT: call
    }
    'father_entry': struct {
        GT: call, 
        DP: int32, 
        AD: array<int32>, 
        GQ: int32, 
        PL: arra

In [119]:
#result_mt.switch_errors.show()
result_mt.filter_rows(result_mt.switch_errors > 0).show()

KeyboardInterrupt: 

In [125]:
dataset.annotate_entries(qq=result_mt[dataset.row_key, dataset.col_key])

2021-11-10 15:19:03 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


In [143]:
mt2.count()

(139596, 430)

2021-11-10 16:16:02 Hail: INFO: Coerced prefix-sorted dataset


In [128]:
input_phased_path='/well/lindgren/UKBIOBANK/nbaya/wes_200k/phase_ukb_wes/data/phased/non_singleton/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz'
input_unphased_path='data/unphased/post-qc/ukb_wes_200k_filtered_chr21.mt'

In [129]:
# get tables
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons

In [130]:
# get trios only
ht = samples.get_fam(app_id=11867, wes_200k_only=False)
trios = ht.filter((ht.PAT != '0') & (ht.MAT != '0'))

# Filter the two matrix tables to the trios defined
mt1 = mt1.filter_cols(hl.is_defined(trios[mt1.s].FID))
mt2 = mt2.filter_cols(hl.is_defined(trios[mt2.s].FID))
    
# load pedigree
fam_path = samples.get_fam_path(app_id=11867,wes_200k_only=False,relateds_only=False)
ped = hl.Pedigree.read(fam_path)

# phase by transmission
trio_dataset = hl.trio_matrix(mt2, pedigree, complete_trios=True)
phased_trio_dataset = hl.experimental.phase_trio_matrix_by_transmission(trio_dataset)
    
# annotate shapeit4 GTs with GTs phased by transmission
mt1 = mt1.annotate_entries(PBT_GT=phased_trio_dataset[mt1.row_key, mt1.col_key].proband_entry.PBT_GT)
mt1 = mt1.filter_entries(hl.is_defined(mt1.PBT_GT))


2021-11-10 16:04:16 Hail: INFO: Reading table without type imputation
  Loading field 'FID' as type str (user-supplied)
  Loading field 'IID' as type str (user-supplied)
  Loading field 'PAT' as type str (user-supplied)
  Loading field 'MAT' as type str (user-supplied)
  Loading field 'SEX' as type str (user-supplied)
  Loading field 'PHEN' as type str (user-supplied)
2021-11-10 16:05:20 Hail: WARN: Found 1 samples with missing sex information (not 1 or 2).
 Missing samples: [{'IID'}]


In [137]:
mt1 = mt1.annotate_rows(switch_errors_count=hl.agg.sum(mt1.PBT_GT != mt1.GT))
mt1 = mt1.annotate_rows(switch_errors=hl.agg.fraction(mt1.PBT_GT != mt1.GT))

In [140]:
mt1.filter_rows(mt1.switch_errors > 0).show()

KeyboardInterrupt: 

In [136]:
mt1.rows().select('switch_errors','switch_errors_count').export()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'switch_errors': float64 
    'switch_errors_count': int64 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


--------------------------------------------------------
Type:
        struct {
        FID: str, 
        PAT: str, 
        MAT: str, 
        SEX: str, 
        PHEN: str
    }
--------------------------------------------------------
Source:
Index:
    ['column']
--------------------------------------------------------


In [39]:
# load data
ht = hl.read_table('data/qc_old/ukb_wes_200k_chr21_variants_unphased.ht').head(1000)
vep = hl.read_table('data/vep/hail/ukb_wes_200k_chr21_vep.ht')

In [40]:
# Annotate QCed data with VEP
ht = ht.annotate(worst_csq_for_variant_canonical = vep[ht.key].vep.worst_csq_for_variant_canonical)

TypeError: 'Float64Expression' object is not callable

In [25]:
ht_singletons = ht.filter(ht.variant_qc.AC[1] == 1)
ht_singletons.group_by(ht_singletons.worst_csq_for_variant_canonical.most_severe_consequence).aggregate(n=hl.agg.count()).show()

2021-11-15 13:45:08 Hail: INFO: Ordering unsorted dataset with network shuffle


,
most_severe_consequence,n
str,int64
"""3_prime_UTR_variant""",14
"""5_prime_UTR_variant""",1
"""frameshift_variant""",10
"""intron_variant""",112
"""missense_variant""",165
"""non_coding_transcript_exon_variant""",5
"""splice_acceptor_variant""",1
"""splice_donor_variant""",5


In [45]:
annotations = analysis.annotation_case_builder(ht.worst_csq_for_variant_canonical, use_loftee = True)

In [46]:
 annotations = analysis.annotation_case_builder(
        ht.worst_csq_for_variant_canonical, use_loftee=True)
ht = ht.annotate(consequence_category=annotations)

In [55]:
ht_cat_all = ht.group_by(
        ht.consequence_category).aggregate(
        n=hl.agg.(ht.variant_qc.AC[1], 0.5))

In [56]:
ht_cat_all.show()

2021-11-15 14:05:40 Hail: INFO: Ordering unsorted dataset with network shuffle


+----------------------+----------+----------+----------+----------+-------+----------+
| consequence_category |   n.mean |  n.stdev |    n.min |    n.max |   n.n |    n.sum |
+----------------------+----------+----------+----------+----------+-------+----------+
| str                  |  float64 |  float64 |  float64 |  float64 | int64 |  float64 |
+----------------------+----------+----------+----------+----------+-------+----------+
| "damaging_missense"  | 2.34e+00 | 2.22e+00 | 0.00e+00 | 1.00e+01 |    44 | 1.03e+02 |
| "non_coding"         | 2.21e+03 | 2.35e+04 | 1.00e+00 | 3.45e+05 |   418 | 9.22e+05 |
| "other_missense"     | 1.00e+03 | 1.57e+04 | 1.00e+00 | 2.82e+05 |   329 | 3.29e+05 |
| "ptv"                | 8.95e+00 | 2.18e+01 | 1.00e+00 | 1.30e+02 |    42 | 3.76e+02 |
| "ptv_LC"             | 3.50e+00 | 5.00e-01 | 3.00e+00 | 4.00e+00 |     4 | 1.40e+01 |
| "synonymous"         | 1.56e+01 | 9.71e+01 | 1.00e+00 | 1.04e+03 |   118 | 1.85e+03 |
| NA                   | 1.46e+03 | 9.58e+03 | 1.00e+00 | 6.50e+04 |    45 | 6.56e+04 |
+----------------------+----------+----------+----------+----------+-------+----------+

In [58]:
ht1 = hl.read_table('data/qc_old/ukb_wes_200k_chr21_variants_unphased.ht').head(1000)
ht2 = hl.read_table('data/qc_old/ukb_wes_200k_chr22_variants_unphased.ht').head(1000)

In [59]:
hts = [ht1, ht2]

In [62]:
ht = hts[0].union(*hts[1:])

In [63]:
ht.count()

2000